# K-means|| distribuito su Dask — Analisi

Notebook di orchestrazione: avvia il cluster, carica il dataset, esegue i test (singola run o benchmark su più combinazioni) e salva i risultati in `./results`.

Moduli usati:
- `kmeans_parallel.py` — algoritmo k-means|| distribuito
- `launch_cluster.py` — avvio/spegnimento del cluster Dask via SSH
- `data_loader.py` — caricamento del dataset
- `benchmark.py` — esecuzione dei test e salvataggio risultati

## 1. Import

In [1]:
import numpy as np
import pandas as pd
from kmeans_parallel import kmeans_parallel
from launch_cluster import launch_cluster, shutdown_cluster
from data_loader import load_dataset
from benchmark import run_single_test, run_benchmark, calculate_inertia

## 2. Parametri configurabili

Modifica qui i valori per cambiare numero di worker, `k`, `l` (oversampling factor) e `r` (numero di round dell'inizializzazione parallela).

In [2]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers

# --- Algoritmo k-means|| ---
K = 500                # numero di cluster finali
L = 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
R = 10                  # numero di round dell'inizializzazione parallela
MAX_ITER_FIT = 10      # iterazioni massime della fase di Lloyd's (fit)
SEED = 42

## 3. Avvio del cluster

In [3]:
cluster, client = launch_cluster(N_WORKERS)
#clientclient.scheduler.address


Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-07-14 14:21:19,517 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:19,516 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-07-14 14:21:19,544 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:19,543 - distributed.scheduler - INFO - State start
2026-07-14 14:21:19,548 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:19,547 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-07-14 14:21:21,726 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:21,730 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.145:42097'
2026-07-14 14:21:21,769 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:21,770 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121:41131'
2026-07-14 14:21:21,803 - distributed.deploy.ssh - INFO - 2026-07-14 14:21:21,805 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.48

Cluster avviato e connessione stabilita con successo!



In [5]:
client.scheduler_info()

{'type': 'Scheduler',
 'id': 'Scheduler-eb1f257f-e8a0-4105-854f-491eb91bd7bb',
 'address': 'tcp://10.67.22.194:8786',
 'services': {'dashboard': 8787},
 'started': 1784038879.2011797,
 'n_workers': 8,
 'total_threads': 64,
 'total_memory': 66574651392,
 'workers': {'tcp://10.67.22.121:45611': {'type': 'Worker',
   'id': 'tcp://10.67.22.121:45611',
   'host': '10.67.22.121',
   'resources': {},
   'local_directory': '/tmp/dask-scratch-space/worker-bpczmyzt',
   'name': 'tcp://10.67.22.121:45611',
   'nthreads': 8,
   'memory_limit': 8321835008,
   'last_seen': 1784038895.6412935,
   'services': {'dashboard': 33683},
   'metrics': {'task_counts': {},
    'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
    'digests_total_since_heartbeat': {'tick-duration': 0.49990200996398926,
     'latency': 0.0013115406036376953},
    'managed_bytes': 0,
    'spilled_bytes': {'memory': 0, 'disk': 0},
    'transfer': {'incoming_bytes': 0,
     'incoming_count': 0,
     'incoming_count_total': 0,
     'outgoing_bytes': 0,
     'outgoing_count': 0,
     'outgoing_count_total': 0},
    'event_loop_interval': 0.01997702121734619,
    'cpu': 2.0,
    'memory': 61349888,
    'time': 1784038895.1431525,
    'host_net_io': {'read_bps': 262.09195024323174,
     'write_bps': 1432.502567741633},
    'host_disk_io': {'read_bps': 0.0, 'write_bps': 0.0},
    'num_fds': 21},
   'status': 'running',
   'nanny': 'tcp://10.67.22.121:41131'},
  'tcp://10.67.22.145:45809': {'type': 'Worker',
   'id': 'tcp://10.67.22.145:45809',
   'host': '10.67.22.145',
   'resources': {},
   'local_directory': '/tmp/dask-scratch-space/worker-tly0hb_c',
   'name': 'tcp://10.67.22.145:45809',
   'nthreads': 8,
   'memory_limit': 8321835008,
   'last_seen': 1784038895.612821,
   'services': {'dashboard': 39053},
   'metrics': {'task_counts': {},
    'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
    'digests_total_since_heartbeat': {'tick-duration': 0.49919557571411133,
     'latency': 0.0013720989227294922},
    'managed_bytes': 0,
    'spilled_bytes': {'memory': 0, 'disk': 0},
    'transfer': {'incoming_bytes': 0,
     'incoming_count': 0,
     'incoming_count_total': 0,
     'outgoing_bytes': 0,
     'outgoing_count': 0,
     'outgoing_count_total': 0},
    'event_loop_interval': 0.02001832962036133,
    'cpu': 2.0,
    'memory': 61145088,
    'time': 1784038895.1184168,
    'host_net_io': {'read_bps': 261.17695515077196,
     'write_bps': 1427.5015258622343},
    'host_disk_io': {'read_bps': 0.0, 'write_bps': 0.0},
    'num_fds': 21},
   'status': 'running',
   'nanny': 'tcp://10.67.22.145:42097'},
  'tcp://10.67.22.187:35779': {'type': 'Worker',
   'id': 'tcp://10.67.22.187:35779',
   'host': '10.67.22.187',
   'resources': {},
   'local_directory': '/tmp/dask-scratch-space/worker-kqxhh0yt',
   'name': 'tcp://10.67.22.187:35779',
   'nthreads': 8,
   'memory_limit': 8321822720,
   'last_seen': 1784038895.3081207,
   'services': {'dashboard': 42951},
   'metrics': {'task_counts': {},
    'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
    'digests_total_since_heartbeat': {'tick-duration': 0.49977779388427734,
     'latency': 0.003498077392578125},
    'managed_bytes': 0,
    'spilled_bytes': {'memory': 0, 'disk': 0},
    'transfer': {'incoming_bytes': 0,
     'incoming_count': 0,
     'incoming_count_total': 0,
     'outgoing_bytes': 0,
     'outgoing_count': 0,
     'outgoing_count_total': 0},
    'event_loop_interval': 0.019998250007629396,
    'cpu': 6.0,
    'memory': 61779968,
    'time': 1784038894.8119843,
    'host_net_io': {'read_bps': 261.3220735729364,
     'write_bps': 1428.2946922001713},
    'host_disk_io': {'read_bps': 0.0, 'write_bps': 0.0},
    'num_fds': 21},
   'status': 'running',
   'nanny': 'tcp://10.67.22.187:43785'},
  'tcp://10.67.22.18:41493': {'type': 'Worker',
   'id': 'tcp://10.67.22.18:41493',
   'host': '10.67.22.18',
   'resources': {},
   'local_directory': '/tmp/dask-scratch-space/worker-dlqu_xqd',
 

## 4. Caricamento del dataset

In [13]:
X = load_dataset("kddcup99", subset="SA", percent10=True)
#troppo grande il 100%
X.shape

Loading dataset...
Dataset preprocessed. Shape: (100655, 38)


(100655, 38)

## 5. Singola run

Esegue una sola combinazione di parametri (quelli definiti nella sezione 2) e stampa costo e tempo di esecuzione.

In [6]:
result, _ = run_single_test(
    client, X,
    k=K, l=L, r=R,
    num_partitions=NUM_PARTITIONS,
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
)
result

 -> Cost: 32024.50 | Time: 42.13s


{'k': 500,
 'l': 250,
 'r': 8,
 'partitions': 16,
 'cost': np.float64(32024.501172765304),
 'time': 42.12557506561279}

## 6. Benchmark multi-combinazione

Testa più combinazioni di `(n_workers, partitions, l_over_k, r)` per diversi valori di `k`, e salva tutto in `./results`. 

**Nota:** il numero di worker effettivamente attivi nel cluster è quello impostato con `launch_cluster` in sezione 3 — la colonna `workers` qui sotto serve solo per etichettare/loggare i risultati, non riavvia il cluster.

In [ ]:
combinations = [
    # (n_workers, num_partitions, l_over_k, r)
    (N_WORKERS, NUM_PARTITIONS+i, 1, R) for i in range(1,50,2)
]

K_VALUES = [1000]

df_results = run_benchmark(
    client, X,
    combinations=combinations,
    k_values=K_VALUES,
    label="kddcup99_benchmark",
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
    averaging_iterations = 10
)
df_results

Testing: k=1000, workers=8, partitions=65, l=1000 (l/k=1), r=10 
 Iterating 10 times.
 -> Cost: 17974.17 | Time: 90.73s
 -> Cost: 17841.91 | Time: 88.78s
 -> Cost: 17905.34 | Time: 87.94s


## 6.1 Analisi dati

Per vedere i risultati ottenuti precedentemente

In [19]:
from benchmark_analysis import BenchmarkAnalyzer

analyzer = BenchmarkAnalyzer(
    data_path="/home/ubuntu/Project/working/results/kddcup99_benchmark_20260714_205025.csv",
    output_dir="/home/ubuntu/Project/working/results/plots",
    facet_cols=["k"],              # una figura per ogni valore (combinazione) di queste colonne
    x_col="partitions",               # variabile sull'asse x
    metrics=["cost", "time"],       # colonne di cui calcolare mean/std e plottare
)
grouped = analyzer.compute_grouped_stats(groupby_cols=["k", "partitions"])
analyzer.print_summary(grouped)
analyzer.plot_all(grouped)

   k  partitions    cost_mean  cost_std  time_mean   time_std  n_runs
1000          65 17919.131558 71.353358 137.874900 122.556448      10
1000          67 17922.735302 55.800857  84.647287   3.800118      10
1000          69 17960.141575 38.033450  82.895011   2.793958      10
1000          71 17920.767241 66.300495  84.256649   1.693807      10
1000          73 17913.972364 46.895644  84.713419   1.377970      10
1000          75 17932.509021 50.659729  83.529865   2.137052      10
1000          77 17927.165965 75.759642  80.110966   1.011836      10
1000          79 17942.269382 63.758669  82.471137   1.845440      10
1000          81 17932.161051 67.931189  82.661148   1.878459      10
1000          83 17942.208339 82.667241  81.995846   1.844266      10
1000          85 17928.294190 55.902298  84.188580   2.591723      10
1000          87 17949.266102 74.106023  82.515809   1.583006      10
1000          89 17906.945090 72.546749  80.900575   2.478579      10
1000          91 179

['/home/ubuntu/Project/working/results/plots/k_1000_partitions.png']

## 7. Spegnimento del cluster

Da eseguire a fine lavoro, o prima di rilanciare `launch_cluster` con un `N_WORKERS` diverso.

In [6]:
shutdown_cluster(cluster, client)

Cluster e client chiusi.
